In [22]:
# Merge the train datasets into one file for easier loading
import os
import pandas as pd
from glob import glob
from tqdm import tqdm



def merge_datasets(data_dir):
    # Get a list of all CSV files in the directory
    csv_files = glob(os.path.join(data_dir, "*.csv"))

    # Initialize an empty list to hold the DataFrames
    dfs = []

    # Loop through the CSV files and read them into DataFrames
    for csv_file in tqdm(csv_files, desc="Loading data"):
        df = pd.read_csv(csv_file)
        dfs.append(df)
    
    # merge all the DataFrames into one by Latitude and Longitude
    # so if df1 has Latitude and Longitude columns, and df2 has Latitude and Longitude columns, then we merge them on those columns
    merged_df = dfs[0]
    for df in dfs[1:]:
        merged_df = pd.merge(merged_df, df, on=["Latitude", "Longitude", "Sample Date"], how="outer")
    
    return merged_df

# training datasets are in the "data/training_data" directory
train_data_dir = "../data/training_data"

trained_df = merge_datasets(train_data_dir)

# Save the merged DataFrame to a new CSV file
output_file = "../data/merged_training_data.csv"
trained_df.to_csv(output_file, index=False)
print(f"Merged training dataset saved to {output_file}")

# testing datasets are in the "data/testing_data" directory
test_data_dir = "../data/testing_data"

tested_df = merge_datasets(test_data_dir)

# Save the merged DataFrame to a new CSV file
output_file = "../data/merged_testing_data.csv"
tested_df.to_csv(output_file, index=False)
print(f"Merged testing dataset saved to {output_file}")

Loading data: 100%|██████████| 3/3 [00:00<00:00, 82.00it/s]


Merged training dataset saved to ../data/merged_training_data.csv


Loading data: 100%|██████████| 2/2 [00:00<00:00, 700.69it/s]

Merged testing dataset saved to ../data/merged_testing_data.csv


In [23]:
# Features to predict
output_vars = ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]

In [24]:
# Check values of data_quality
print("Unique data_quality values:")
print(trained_df["data_quality"].unique())


Unique data_quality values:
['good' 'fair']


In [25]:
import numpy as np
# Create a method to preprocess the data for modeling
def preprocess_data(df):
    # One hot encode data_quality
    df = pd.get_dummies(df, columns=["data_quality"], prefix="data_quality")

    # Split date into day, month, year
    df["Sample Date"] = pd.to_datetime(df["Sample Date"], dayfirst=True, errors='coerce')
    df["day"] = df["Sample Date"].dt.day
    df["month"] = df["Sample Date"].dt.month
    df["year"] = df["Sample Date"].dt.year
    df.drop(columns=["Sample Date", "coastal"], inplace=True)

    return df

# Preprocess the training data
trained_df = preprocess_data(trained_df)
# Preprocess the testing data
tested_df = preprocess_data(tested_df)
# After get_dummies on both, align columns
tested_df = tested_df.reindex(columns=trained_df.columns, fill_value=0)

# Cyclical month encoding
trained_df['month_sin'] = np.sin(2 * np.pi * trained_df['month'] / 12)
trained_df['month_cos'] = np.cos(2 * np.pi * trained_df['month'] / 12)
tested_df['month_sin'] = np.sin(2 * np.pi * tested_df['month'] / 12)
tested_df['month_cos'] = np.cos(2 * np.pi * tested_df['month'] / 12)

# Spatial clusters — fit on train only, apply to both
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=20, random_state=42)
trained_df['spatial_cluster'] = kmeans.fit_predict(trained_df[['Latitude', 'Longitude']])
tested_df['spatial_cluster'] = kmeans.predict(tested_df[['Latitude', 'Longitude']])

# Impute remaining NaNs with median per column
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')

# Impute columns except Latitude, Longitude, day, month, year
features = trained_df.drop(columns=output_vars).columns
impute_features = features.difference(["Latitude", "Longitude", "day", "month", "year", "spatial_cluster"])
imputer.fit(trained_df[impute_features])
trained_df[impute_features] = imputer.transform(trained_df[impute_features])
tested_df[impute_features] = imputer.transform(tested_df[impute_features])

# After imputation, BEFORE scaling, add:
for df in [trained_df, tested_df]:
    df['turbidity_x_rain'] = df['TurbidityIndex'] * df['ppt']
    df['runoff_x_soil']    = df['q'] * df['soil']
    df['ndvi_x_rain']      = df['NDVI'] * df['ppt']

# Run standard scaling on the features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# scale all features except Latitude, Longitude, day, month, year
# Explicitly exclude targets from scaling
features_to_scale = [c for c in trained_df.columns 
                     if c not in output_vars + ["Latitude", "Longitude", "day", "month", "year", "coastal", "spatial_cluster"]]

trained_df[features_to_scale] = scaler.fit_transform(trained_df[features_to_scale])
tested_df[features_to_scale] = scaler.transform(tested_df[features_to_scale])

/var/folders/yk/c0cfqj715njfdnphhzw8m5p00000gn/T/ipykernel_7380/701273665.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["Sample Date"] = pd.to_datetime(df["Sample Date"], dayfirst=True, errors='coerce')
/var/folders/yk/c0cfqj715njfd

In [26]:
print("Any targets scaled?", any(t in features_to_scale for t in output_vars))
print("Spatial cluster scaled?", 'spatial_cluster' in features_to_scale)
print("Train NaNs:", trained_df.isnull().sum().sum())
print("Test NaNs:", tested_df.isnull().sum().sum())

Any targets scaled? False
Spatial cluster scaled? False
Train NaNs: 0
Test NaNs: 0


In [27]:
tested_df.columns

Index(['Latitude', 'Longitude', 'pet', 'aet', 'def', 'ppt', 'q', 'soil',
       'pdsi', 'tmax', 'tmin', 'vap', 'swe', 'temp_range', 'aridity_index',
       'evap_fraction', 'water_balance', 'Total Alkalinity',
       'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'blue',
       'green', 'red', 'nir', 'swir16', 'swir22', 'NDVI', 'EVI', 'SAVI',
       'ARVI', 'GNDVI', 'RDVI', 'NDWI', 'MNDWI', 'AWEInsh', 'AWEIsh', 'BSI',
       'NDBI', 'UI', 'NBR', 'ClayMinerals', 'TurbidityIndex',
       'ChlorophyllIndex', 'NIR_Red_Ratio', 'NDMI', 'cloud_cover',
       'data_quality_fair', 'data_quality_good', 'day', 'month', 'year',
       'month_sin', 'month_cos', 'spatial_cluster', 'turbidity_x_rain',
       'runoff_x_soil', 'ndvi_x_rain'],
      dtype='object')

In [28]:
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
import numpy as np

log_targets = ['Dissolved Reactive Phosphorus']
targets = output_vars

transforms = {
    'Total Alkalinity': 'none',
    'Electrical Conductance': 'none',
    'Dissolved Reactive Phosphorus': 'log1p'
}

models = {}
predictions = {}
val_r2s = {}

X = trained_df.drop(columns=targets)
y_all = trained_df[targets].copy()

for target in log_targets:
    y_all[target] = np.log1p(y_all[target])

X_train, X_val = train_test_split(X, test_size=0.2, random_state=42)

for target in targets:
    print(f"\n{'='*50}")
    print(f"Training model for: {target}")

    y_train = y_all.loc[X_train.index, target]
    y_val   = y_all.loc[X_val.index,   target]

    grid_search = GridSearchCV(
        estimator=LGBMRegressor(random_state=42, verbose=-1),
        param_grid={
            'n_estimators':     [200, 500],
            'num_leaves':       [31, 63, 127],  # key LGBM param
            'max_depth':        [-1, 7],         # -1 = no limit
            'learning_rate':    [0.05, 0.1],
            'subsample':        [0.8],
            'colsample_bytree': [0.8],
            'min_child_samples':[10, 20],        # regularization
        },
        cv=3,
        scoring='r2',
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X_train, y_train)
    model = grid_search.best_estimator_

    y_pred_transformed = model.predict(X_val)
    if transforms[target] == 'log1p':
        y_pred         = np.expm1(y_pred_transformed)
        y_val_original = np.expm1(y_val)
    else:
        y_pred         = y_pred_transformed
        y_val_original = y_val

    r2 = r2_score(y_val_original, y_pred)
    print(f"Best params: {grid_search.best_params_}")
    print(f"Validation R²: {r2:.4f}")

    models[target]  = model
    val_r2s[target] = r2

    X_test = tested_df.drop(columns=targets, errors='ignore')
    raw_preds = model.predict(X_test)
    predictions[target] = np.expm1(raw_preds) if transforms[target] == 'log1p' else raw_preds

print(f"\nMean Validation R²: {sum(val_r2s.values()) / len(val_r2s):.4f}")
for target, r2 in val_r2s.items():
    print(f"  {target}: {r2:.4f}")


Training model for: Total Alkalinity
Fitting 3 folds for each of 48 candidates, totalling 144 fits


/var/folders/yk/c0cfqj715njfdnphhzw8m5p00000gn/T/ipykernel_7380/224266317.py:23: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  y_all[target] = np.log1p(y_all[target])
/Users/shreyasriram/personal_projects/ey/.venv/lib/python3.14/site-package

Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 10, 'n_estimators': 500, 'num_leaves': 63, 'subsample': 0.8}
Validation R²: 0.8423

Training model for: Electrical Conductance
Fitting 3 folds for each of 48 candidates, totalling 144 fits


/Users/shreyasriram/personal_projects/ey/.venv/lib/python3.14/site-packages/numpy/ma/core.py:2885: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_samples': 10, 'n_estimators': 500, 'num_leaves': 127, 'subsample': 0.8}
Validation R²: 0.8639

Training model for: Dissolved Reactive Phosphorus
Fitting 3 folds for each of 48 candidates, totalling 144 fits


/Users/shreyasriram/personal_projects/ey/.venv/lib/python3.14/site-packages/numpy/ma/core.py:2885: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 10, 'n_estimators': 200, 'num_leaves': 63, 'subsample': 0.8}
Validation R²: 0.6369

Mean Validation R²: 0.7810
  Total Alkalinity: 0.8423
  Electrical Conductance: 0.8639
  Dissolved Reactive Phosphorus: 0.6369


In [29]:
# Run best model on test set and save predictions
test_predictions = pd.DataFrame(predictions)
X_test = tested_df.copy()  # Make a copy of the test DataFrame to avoid modifying the original
test_predictions['Latitude'] = X_test['Latitude']
test_predictions['Longitude'] = X_test['Longitude']
# Change day, month to two digit format for better readability
test_predictions['Sample Date'] = (
    X_test['day'].astype(str).str.zfill(2) + '-' + 
    X_test['month'].astype(str).str.zfill(2) + '-' + 
    X_test['year'].astype(str)
)
for target in targets:
    test_predictions[target] = predictions[target]

test_predictions = test_predictions[["Latitude", "Longitude", "Sample Date"] + targets] 
output_file = "../data/modeling/test_predictions_final.csv"
test_predictions.to_csv(output_file, index=False)